In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import xgboost as xgb
from sklearn.ensemble import GradientBoostingClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

from pathlib import Path
import sys

sys.path.append('..')

from src.preprocessing import load_processed_data

In [31]:
X_train, X_test, X_val, y_train, y_test, y_val = load_processed_data()
X_train.head()

Label Encoding applied to categorical features in X and target variable y.


,Age,BusinessTravel,DailyRate,Department,DistanceFromHome,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,...,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany
1097,24,2,350,1,21,5,1,1551,3,57,...,0,14,3,2,80,3,2,3,3,1
727,18,0,287,1,5,1,1,1012,2,73,...,0,15,3,4,80,0,0,2,3,0
254,29,2,1247,2,20,2,1,349,4,45,...,0,14,3,4,80,1,10,2,3,3
1175,39,2,492,1,12,3,1,1654,4,66,...,0,21,4,3,80,0,7,3,3,5
1341,31,2,311,1,20,1,1,1881,2,89,...,0,11,3,1,80,1,10,2,3,10


# Model

Decision Tree

In [32]:
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)

y_train_pred = decision_tree.predict(X_train)
y_test_pred = decision_tree.predict(X_test)

y_train_accuracy = accuracy_score(y_train, y_train_pred)
y_test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {y_train_accuracy}")
print(f"Testing Accuracy: {y_test_accuracy}")


Training Accuracy: 1.0
Testing Accuracy: 0.7891156462585034


Random Forest

In [33]:
random_forest = RandomForestClassifier(random_state=42)
random_forest.fit(X_train, y_train)

y_train_pred = random_forest.predict(X_train)
y_test_pred = random_forest.predict(X_test)

y_train_accuracy = accuracy_score(y_train, y_train_pred)
y_test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {y_train_accuracy}")
print(f"Testing Accuracy: {y_test_accuracy}")


Training Accuracy: 1.0
Testing Accuracy: 0.8707482993197279


Boosting

XG Boost

In [34]:
xgb_classifier = xgb.XGBClassifier(random_state=42)
xgb_classifier.fit(X_train, y_train)

y_train_pred = xgb_classifier.predict(X_train)
y_test_pred = xgb_classifier.predict(X_test)

y_train_accuracy = accuracy_score(y_train, y_train_pred)
y_test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {y_train_accuracy}")
print(f"Testing Accuracy: {y_test_accuracy}")

Training Accuracy: 1.0
Testing Accuracy: 0.8571428571428571


Gradient Boosting Classifier

In [35]:
gbc = GradientBoostingClassifier(random_state=42)
gbc.fit(X_train, y_train)

y_train_pred = gbc.predict(X_train)
y_test_pred = gbc.predict(X_test)

y_train_accuracy = accuracy_score(y_train, y_train_pred)
y_test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {y_train_accuracy}")
print(f"Testing Accuracy: {y_test_accuracy}")

# X_train.columns

Training Accuracy: 0.9591836734693877
Testing Accuracy: 0.8843537414965986


> all

Training Accuracy: 0.9583333333333334

Testing Accuracy: 0.8843537414965986

> drop education

Training Accuracy: 0.9600340136054422

Testing Accuracy: 0.8775510204081632

> drop gender

Training Accuracy: 0.9574829931972789

Testing Accuracy: 0.8775510204081632

> drop gender and education

Training Accuracy: 0.9583333333333334

Testing Accuracy: 0.8843537414965986

> drop 'Education', 'Gender', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager' ✅

Training Accuracy: 0.9591836734693877

Testing Accuracy: 0.8843537414965986

> drop 'Education', 'Gender', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'Hourly Rate'

Training Accuracy: 0.9532312925170068

Testing Accuracy: 0.8775510204081632

> drop 'Education', 'Gender', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager', 'RelationshipSatisfaction'

Training Accuracy: 0.9591836734693877

Testing Accuracy: 0.8775510204081632

In [36]:
# import pickle

# pickle.dump(gbc, open('attrition_model.pkl', 'wb'))
# pickle.dump(le, open('label_encoder.pkl', 'wb'))


catboost

In [37]:
# Initialize CatBoostClassifier
# For categorical features, CatBoost handles them automatically, but we need to identify them.
# First, let's get the names of categorical features from X_train before label encoding.
# Note: In this notebook, label encoding was applied *before* this point, so CatBoost will treat them as numerical.
# If you wanted CatBoost to handle them as true categorical features, encoding would need to be done after splitting, and then specified with cat_features parameter.

catboost_model = CatBoostClassifier(random_state=42, verbose=0,  # verbose=0 to suppress output during training
                                    # For simplicity, we are letting CatBoost treat the already label-encoded features as numerical for now.
                                    # If the original categorical columns were preserved, we would use:
                                    # cat_features=[col for col in X.select_dtypes(include='object').columns if col not in ['Attrition']],
                                   )

# Train the model
catboost_model.fit(X_train, y_train)

# Make predictions on training and test sets
y_train_pred_cb = catboost_model.predict(X_train)
y_test_pred_cb = catboost_model.predict(X_test)

# Calculate accuracy scores
y_train_accuracy_cb = accuracy_score(y_train, y_train_pred_cb)
y_test_accuracy_cb = accuracy_score(y_test, y_test_pred_cb)

print(f"CatBoost Training Accuracy: {y_train_accuracy_cb}")
print(f"CatBoost Testing Accuracy: {y_test_accuracy_cb}")

# Optionally, print classification report for more detailed metrics
print("\nClassification Report for CatBoost on Test Set:")
print(classification_report(y_test, y_test_pred_cb))

CatBoost Training Accuracy: 0.9821428571428571
CatBoost Testing Accuracy: 0.8639455782312925

Classification Report for CatBoost on Test Set:
              precision    recall  f1-score   support

           0       0.87      0.99      0.93       125
           1       0.75      0.14      0.23        22

    accuracy                           0.86       147
   macro avg       0.81      0.56      0.58       147
weighted avg       0.85      0.86      0.82       147



# Hyperperameter Tuning

XG Boost

In [38]:
# param_grid_xgb = {
#     'n_estimators': [100, 200, 300],
#     'learning_rate': [0.01, 0.1, 0.2, 0.5],
#     'max_depth': [3, 5, 7, 8,9,10,11,12,13,14,15],
#     'subsample': [0.6, 0.8, 1.0],
# }

# xgb_classifier = xgb.XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')
# grid_search_xgb = GridSearchCV(estimator=xgb_classifier, param_grid=param_grid_xgb, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
# grid_search_xgb.fit(X_train, y_train)

# print("Best parameters for XGBoost Classifier:", grid_search_xgb.best_params_)
# print("Best accuracy for XGBoost Classifier (cross-validation):", grid_search_xgb.best_score_)

Fitting 5 folds for each of 396 candidates, totalling 1980 fits
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [20:52:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782:
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
Best parameters for XGBoost Classifier: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0}
Best accuracy for XGBoost Classifier (cross-validation): 0.8631013342949874

In [39]:
# # Evaluate the best XGBoost Classifier on the test set
# best_xgb_classifier = grid_search_xgb.best_estimator_
# y_test_pred_xgb_tuned = best_xgb_classifier.predict(X_test)
# y_test_accuracy_xgb_tuned = accuracy_score(y_test, y_test_pred_xgb_tuned)

# print(f"Test Accuracy of Tuned XGBoost Classifier: {y_test_accuracy_xgb_tuned}")

Test Accuracy of Tuned XGBoost Classifier: 0.8775510204081632

Gradient Boosting Classifier

In [40]:
# param_grid_gb = {
#     'n_estimators': [100, 200, 300],
#     'learning_rate': [0.01, 0.1, 0.2],
#     'max_depth': [3, 4, 5],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'subsample': [0.8, 1.0],
# }

# gbc = GradientBoostingClassifier(random_state=42)
# grid_search_gb = GridSearchCV(estimator=gbc, param_grid=param_grid_gb, cv=5, scoring='accuracy', n_jobs=-1)
# grid_search_gb.fit(X_train, y_train)


# print("Best parameters for Gradient Boosting Classifier:", grid_search_gb.best_params_)
# print("Best accuracy for Gradient Boosting Classifier:", grid_search_gb.best_score_)

Best parameters for Gradient Boosting Classifier: {'learning_rate': 0.1, 'max_depth': 3, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100, 'subsample': 0.8}
Best accuracy for Gradient Boosting Classifier: 0.8707753335737468

In [41]:
# # Evaluate the best Gradient Boosting Classifier on the test set
# best_gb_classifier = grid_search_gb.best_estimator_
# y_test_pred_tuned = best_gb_classifier.predict(X_test)
# y_test_accuracy_tuned = accuracy_score(y_test, y_test_pred_tuned)

# print(f"Test Accuracy of Tuned Gradient Boosting Classifier: {y_test_accuracy_tuned}")

Test Accuracy of Tuned Gradient Boosting Classifier: 0.8707482993197279

### Hyperparameter Tuning for Gradient Boosting Classifier using RandomizedSearchCV

In [42]:

# # Define the parameter distribution for RandomizedSearchCV
# param_dist_gb = {
#     'n_estimators': randint(50, 400),
#     'learning_rate': uniform(0.01, 0.2),
#     'max_depth': randint(3, 10),
#     'min_samples_split': randint(2, 20),
#     'min_samples_leaf': randint(1, 10),
#     'subsample': uniform(0.6, 0.4),
# }

# gbc_random = GradientBoostingClassifier(random_state=42)

# # Initialize RandomizedSearchCV
# # n_iter specifies the number of parameter settings that are sampled.
# # cv is the number of folds for cross-validation.
# random_search_gb = RandomizedSearchCV(
#     estimator=gbc_random,
#     param_distributions=param_dist_gb,
#     n_iter=50,  # Number of parameter settings that are sampled
#     cv=5,       # 5-fold cross-validation
#     scoring='accuracy',
#     n_jobs=-1,  # Use all available cores
#     random_state=42,
#     verbose=1
# )

# # Fit RandomizedSearchCV to the training data
# random_search_gb.fit(X_train, y_train)

# print("Best parameters for Gradient Boosting Classifier (RandomizedSearchCV):", random_search_gb.best_params_)
# print("Best accuracy for Gradient Boosting Classifier (cross-validation, RandomizedSearchCV):", random_search_gb.best_score_)

### Evaluating the Tuned Gradient Boosting Classifier

In [43]:
# # Evaluate the best Gradient Boosting Classifier from RandomizedSearchCV on the test set
# best_gb_classifier_random = random_search_gb.best_estimator_
# y_test_pred_random_tuned = best_gb_classifier_random.predict(X_test)
# y_test_accuracy_random_tuned = accuracy_score(y_test, y_test_pred_random_tuned)

# print(f"Test Accuracy of Tuned Gradient Boosting Classifier (RandomizedSearchCV): {y_test_accuracy_random_tuned}")

# # Optionally, print classification report for more detailed metrics
# print("\nClassification Report for Tuned GBC (RandomizedSearchCV) on Test Set:")
# print(classification_report(y_test, y_test_pred_random_tuned))